In [ ]:
# TODO: make functions less verbose for data creation
# TODO: clean import statments

In [ ]:
 # this is a good little tutorial to understand basics of pyspark  
# https://domino.ai/blog/principal-component-analysis-pca-on-large-neuroimaging-datasets-using-pyspark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [19]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

In [20]:
# Spark is a library that distributes the load of computation/ram very efficiently and evenly :)

In [21]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

Stopping existing Spark context...
Previous Spark context stopped successfully


In [56]:
# Set environment variables
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# Create new session with explicit local binding
spark = SparkSession.builder \
    .appName("EEG_Analysis") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .master("local[*]") \
    .getOrCreate()


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

New Spark session created successfully


25/04/07 02:52:42 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/04/07 02:52:42 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [26]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context


In [27]:
subject_df = load_subjects_df(spark) #this is the .tsv with the information of all the participants

In [29]:
%%time
#Example below is how to get a single subject  and extract its features
sub1 = (
    subject_df
    .filter((subject_df.SubjectID == "sub-001"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)
sub1.show()

/Users/admin/neuro-venv/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
function ran                                                        (0 + 1) / 1]
Processing subject sub-001
processSub sub-001
subPath sub-001
Path handed: /Users/admin/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Got 398 epochs for sub-001
<class 'mne.epochs.Epochs'>
Epoch 0
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>
Epoch 1
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>
Total rows collected: 37810
Returning DataFrame with 37810 rows
Column names from schema: ['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']
4.9767937660217285
                                                                                

+---------+-------+--------+---------+--------------------+
|SubjectID|EpochID|WaveBand|Electrode|               Power|
+---------+-------+--------+---------+--------------------+
|  sub-001|   ep-0|   Delta|      Fp1| 0.08257892642542033|
|  sub-001|   ep-0|   Theta|      Fp1|0.005611645685714301|
|  sub-001|   ep-0|   Alpha|      Fp1|0.001143220388171...|
|  sub-001|   ep-0|    Beta|      Fp1|1.958040080323613...|
|  sub-001|   ep-0|   Total|      Fp1|0.011235955056179773|
|  sub-001|   ep-0|   Delta|      Fp2| 0.08389220643958975|
|  sub-001|   ep-0|   Theta|      Fp2|0.004672355992737972|
|  sub-001|   ep-0|   Alpha|      Fp2|9.638188967593915E-4|
|  sub-001|   ep-0|    Beta|      Fp2|1.768820461211869...|
|  sub-001|   ep-0|   Total|      Fp2|0.011235955056179771|
|  sub-001|   ep-0|   Delta|       F3| 0.08212274846138015|
|  sub-001|   ep-0|   Theta|       F3|0.005595318986261473|
|  sub-001|   ep-0|   Alpha|       F3|0.001112336388491...|
|  sub-001|   ep-0|    Beta|       F3|2.

In [36]:
%%time

# this is the magic of pyspark's distributed system: What would take 50 minutes single threaded takes around 4.5
# I did a similiar optimization with joblib where I would multiprocess this steap and it would take around 7

# What's nice is that we can process whole groups pretty easily :)
group_a_spark_df = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

group_c_spark_df = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

# from pyspark.sql import functions as F 

# Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
result_group_a = result_group_a.persist()
result_group_c = result_group_c.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")

/Users/admin/neuro-venv/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
25/04/07 01:55:08 WARN CacheManager: Asked to cache already cached data.
25/04/07 01:55:08 WARN CacheManager: Asked to cache already cached data.


Processed 1856870 records for Alzheimer's group
Processed 1543465 records for Control group
CPU times: user 20.4 ms, sys: 15.6 ms, total: 36.1 ms
Wall time: 541 ms


In [37]:
result_group_a.columns

['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']

In [32]:
#Since we doni't want to recreate the data all the time, lets save it and I will see you in Example_Data_Processing

In [38]:
type(group_a_spark_df)

pyspark.sql.dataframe.DataFrame

In [39]:
group_a_pandas_df = group_a_spark_df.toPandas() # see here, spark has its own data frame type with lots of its own functions
group_c_pandas_df = group_c_spark_df.toPandas() # Each .pkl is around 50mb last time I checked


In [40]:
group_a_pandas_df.to_pickle("features_alz_example.pkl") # pkl is a way to store python dataframes, its nice
group_c_pandas_df.to_pickle("features_cntrl_example.pkl") # pkl is a way to store python dataframes, its nice

In [47]:
# This is how we would load the .pkl's back in 
# Step 1: Load back into pandas
group_a_pandas_df_loaded = pd.read_pickle("features_alz_example.pkl")
group_c_pandas_df_loaded = pd.read_pickle("features_cntrl_example.pkl")

# Step 2: Convert to Spark DataFrames
group_a_spark_df_loaded = spark.createDataFrame(group_a_pandas_df_loaded)
group_c_spark_df_loaded = spark.createDataFrame(group_c_pandas_df_loaded)

In [48]:
type(group_a_spark_df)

pyspark.sql.dataframe.DataFrame

In [49]:
if group_a_spark_df_loaded.exceptAll(group_a_spark_df).isEmpty(): 
    print("Correctly pkl'd and correcfly loaded into pyspark object")

25/04/07 02:08:24 WARN TaskSetManager: Stage 68 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Correctly pkl'd and correcfly loaded into pyspark object


In [55]:
group_c_spark_df_loaded.select("SubjectID").distinct().orderBy("SubjectID").show(truncate=False)

25/04/07 02:22:28 WARN TaskSetManager: Stage 94 contains a task of very large size (6561 KiB). The maximum recommended task size is 1000 KiB.


+---------+
|SubjectID|
+---------+
|sub-037  |
|sub-038  |
|sub-039  |
|sub-040  |
|sub-041  |
|sub-042  |
|sub-043  |
|sub-044  |
|sub-045  |
|sub-046  |
|sub-047  |
|sub-048  |
|sub-049  |
|sub-050  |
|sub-051  |
|sub-052  |
|sub-053  |
|sub-054  |
|sub-055  |
|sub-056  |
+---------+
only showing top 20 rows

